In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    f1_score
)

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import EarlyStopping

In [2]:
data = pd.read_csv(
    "../data/features.csv",
    index_col=0,
    parse_dates=True
)


data.head()

,Close,Log_Return,Return,SMA20,SMA50,SMA100,SMA200,EMA20,Momentum10,Momentum20,...,Lag2,Lag5,Lag10,Price_SMA20,Price_SMA50,UpperBand,LowerBand,RSI,MACD,MACD_Signal
Date,,,,,,,,,,,,,,,,,,,,,
2015-10-16,24.883137,-0.007358,-0.007331,25.046606,25.176153,26.631793,27.035149,25.013353,0.147905,-0.540051,...,-0.014235,0.023645,0.007274,0.993473,0.988361,25.889497,24.203715,45.695046,-0.136604,-0.159600
2015-10-19,25.037760,0.006195,0.006214,25.007615,25.159167,26.588194,27.039375,25.015677,0.212889,-0.779835,...,0.014861,-0.004649,0.003617,1.001205,0.995174,25.768494,24.246735,59.838000,-0.118958,-0.151471
2015-10-20,25.494896,0.018093,0.018258,25.011760,25.132500,26.552512,27.049294,25.061317,0.551258,0.082909,...,-0.007358,0.001701,0.004773,1.019316,1.014419,25.782752,24.240768,62.073745,-0.067310,-0.134639
2015-10-21,25.492657,-0.000088,-0.000088,25.005486,25.133710,26.516227,27.059191,25.102397,0.667786,-0.125488,...,0.006195,-0.014235,-0.004773,1.019483,1.014281,25.757509,24.253462,65.300136,-0.026256,-0.112962
2015-10-22,25.882586,0.015180,0.015296,25.011088,25.134875,26.485136,27.069389,25.176701,1.344553,0.112059,...,0.018093,0.014861,-0.011622,1.034844,1.029748,25.788357,24.233819,67.534311,0.037312,-0.082907


In [3]:
features = [

    "Log_Return",
    "Volatility20",
    "Momentum10",
    "RSI",
    "MACD",
    "SMA20",
    "SMA50"

]


dataset = data[features].copy()

dataset = dataset.dropna()


dataset.head()

,Log_Return,Volatility20,Momentum10,RSI,MACD,SMA20,SMA50
Date,,,,,,,
2015-10-16,-0.007358,0.013340,0.147905,45.695046,-0.136604,25.046606,25.176153
2015-10-19,0.006195,0.012893,0.212889,59.838000,-0.118958,25.007615,25.159167
2015-10-20,0.018093,0.013142,0.551258,62.073745,-0.067310,25.011760,25.132500
2015-10-21,-0.000088,0.013009,0.667786,65.300136,-0.026256,25.005486,25.133710
2015-10-22,0.015180,0.013399,1.344553,67.534311,0.037312,25.011088,25.134875


In [4]:
# 5-day forward return

future_return = (
    dataset["Log_Return"]
    .rolling(5)
    .sum()
    .shift(-5)
)


# πρόσφατο regime

rolling_mean = (
    dataset["Log_Return"]
    .rolling(60)
    .mean()
)


# target:
# 1 = outperformance
# 0 = underperformance

dataset["Target"] = (
    future_return > rolling_mean
).astype(int)


dataset = dataset.dropna()


dataset["Target"].value_counts()

Target
1    1510
0    1192
Name: count, dtype: int64

In [5]:
X_data = dataset[features]

y_data = dataset["Target"]

In [6]:
from sklearn.preprocessing import MinMaxScaler


scaler = MinMaxScaler()


X_scaled = scaler.fit_transform(
    X_data
)

In [7]:
def create_sequences_classification(
    X,
    y,
    window=60
):

    Xs = []
    ys = []


    for i in range(
        window,
        len(X)
    ):

        Xs.append(
            X[i-window:i]
        )

        ys.append(
            y.iloc[i]
        )


    return (
        np.array(Xs),
        np.array(ys)
    )

In [8]:
X, y = create_sequences_classification(
    X_scaled,
    y_data,
    window=60
)


print(X.shape)
print(y.shape)

(2642, 60, 7)
(2642,)


In [9]:
split = int(
    len(X)*0.8
)


X_train = X[:split]

X_test = X[split:]


y_train = y[:split]

y_test = y[split:]

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM,
    Dense,
    Dropout
)


model = Sequential()


model.add(
    LSTM(
        32,
        input_shape=(
            X_train.shape[1],
            X_train.shape[2]
        )
    )
)


model.add(
    Dropout(0.3)
)


model.add(
    Dense(
        1,
        activation="sigmoid"
    )
)


model.compile(

    optimizer="adam",

    loss="binary_crossentropy",

    metrics=[
        "accuracy"
    ]

)


model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,153 (20.13 KB)

 Trainable params: 5,153 (20.13 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
from tensorflow.keras.callbacks import EarlyStopping


early_stop = EarlyStopping(
    patience=10,
    restore_best_weights=True
)


history = model.fit(

    X_train,

    y_train,

    epochs=100,

    batch_size=32,

    validation_split=0.1,

    callbacks=[
        early_stop
    ],

    verbose=1
)

Epoch 1/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.5671 - loss: 0.6832 - val_accuracy: 0.5283 - val_loss: 0.6937
Epoch 2/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.5850 - loss: 0.6790 - val_accuracy: 0.5283 - val_loss: 0.6928
Epoch 3/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.5865 - loss: 0.6776 - val_accuracy: 0.4811 - val_loss: 0.6918
Epoch 4/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5871 - loss: 0.6754 - val_accuracy: 0.5283 - val_loss: 0.6934
Epoch 5/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.5886 - loss: 0.6750 - val_accuracy: 0.5283 - val_loss: 0.6929
Epoch 6/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.6002 - loss: 0.6729 - val_accuracy: 0.4811 - val_loss: 0.6937
Epoch 7/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.5902 - loss: 0.6758 - val_accuracy: 0.4953 - val_loss: 0.6961
Epoch 8/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5976 - loss: 0.6711 - val_accuracy: 0.

In [12]:
probabilities = model.predict(
    X_test
)


probabilities[:10]

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step


array([[0.5212763 ],
       [0.5198176 ],
       [0.5193951 ],
       [0.51703215],
       [0.51928425],
       [0.5206718 ],
       [0.52210915],
       [0.52228916],
       [0.5231755 ],
       [0.52203894]], dtype=float32)

In [13]:
from sklearn.metrics import (
    f1_score,
    accuracy_score
)


thresholds = np.arange(
    0.40,
    0.65,
    0.01
)


best_threshold = 0

best_f1 = 0


for t in thresholds:

    preds = (
        probabilities > t
    ).astype(int)


    f1 = f1_score(
        y_test,
        preds
    )


    print(
        round(t,2),
        f1
    )


    if f1 > best_f1:

        best_f1 = f1
        best_threshold = t



print(
    "Best Threshold:",
    best_threshold
)

print(
    "Best F1:",
    best_f1
)

0.4 0.6986469864698647
0.41 0.6986469864698647
0.42 0.6986469864698647
0.43 0.6986469864698647
0.44 0.6986469864698647
0.45 0.6925
0.46 0.6871794871794872
0.47 0.6739427012278308
0.48 0.6191950464396285
0.49 0.5321428571428571
0.5 0.42437923250564336
0.51 0.1924198250728863
0.52 0.027210884353741496
0.53 0.0
0.54 0.0
0.55 0.0
0.56 0.0
0.57 0.0
0.58 0.0
0.59 0.0
0.6 0.0
0.61 0.0
0.62 0.0
0.63 0.0
0.64 0.0
Best Threshold: 0.4
Best F1: 0.6986469864698647


In [14]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    confusion_matrix
)


predictions = (
    probabilities > best_threshold
).astype(int)


print(
    "Accuracy:",
    accuracy_score(
        y_test,
        predictions
    )
)


print(
    "Precision:",
    precision_score(
        y_test,
        predictions
    )
)


print(
    "Recall:",
    recall_score(
        y_test,
        predictions
    )
)


print(
    confusion_matrix(
        y_test,
        predictions
    )
)

Accuracy: 0.5368620037807184
Precision: 0.5368620037807184
Recall: 1.0
[[  0 245]
 [  0 284]]
